# Gold Price Analysis (2023-2026)
**Why gold dropped from $5,400 to $4,300 per ounce in just three weeks**

In [ ]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14,7)

In [ ]:
# Connect to SQL database
conn = sqlite3.connect('gold_prices.db')  # replace with your DB path
query = '''
SELECT date, price_usd
FROM gold_daily
WHERE date >= '2023-01-01'
ORDER BY date ASC;
'''
gold_df = pd.read_sql(query, conn)
gold_df['date'] = pd.to_datetime(gold_df['date'])

In [ ]:
# Calculate EMAs
gold_df['EMA_50'] = gold_df['price_usd'].ewm(span=50, adjust=False).mean()
gold_df['EMA_100'] = gold_df['price_usd'].ewm(span=100, adjust=False).mean()
gold_df['EMA_200'] = gold_df['price_usd'].ewm(span=200, adjust=False).mean()

In [ ]:
# Calculate RSI
def compute_rsi(data, window=14):
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    RS = gain / loss
    RSI = 100 - (100 / (1 + RS))
    return RSI

gold_df['RSI_14'] = compute_rsi(gold_df['price_usd'])

In [ ]:
# Plot Gold Price and EMAs
plt.figure(figsize=(16,8))
plt.plot(gold_df['date'], gold_df['price_usd'], label='Gold Price USD', color='gold')
plt.plot(gold_df['date'], gold_df['EMA_50'], label='EMA 50', linestyle='--')
plt.plot(gold_df['date'], gold_df['EMA_100'], label='EMA 100', linestyle='--')
plt.plot(gold_df['date'], gold_df['EMA_200'], label='EMA 200', linestyle='--')
plt.title('Gold Price in USD (2023-2026)')
plt.xlabel('Date')
plt.ylabel('Price USD')
plt.legend()
plt.show()

In [ ]:
# Plot RSI
plt.figure(figsize=(16,4))
plt.plot(gold_df['date'], gold_df['RSI_14'], label='RSI 14', color='purple')
plt.axhline(30, color='red', linestyle='--', label='Oversold')
plt.axhline(70, color='green', linestyle='--', label='Overbought')
plt.title('Gold RSI (14-Day)')
plt.xlabel('Date')
plt.ylabel('RSI')
plt.legend()
plt.show()